<a href="https://colab.research.google.com/github/Diwash17/FlyRank-AI-Assentment-and-Capstone/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> ⚠️ **Before you rely on any query below**: this notebook was drafted without direct access to the gated warehouse schema. Column names marked `# VERIFY:` are educated guesses based on naming patterns visible in `dim_clients` (e.g. `has_gsc_access`, `has_ga4_access`). Run the `DESCRIBE` cell in Section 1 first, and find/replace any names that don't match your actual `dim_content` / `fact_content_daily_performance` / `fact_content_query_90d` schema before running the rest.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Lane: 3 — Structured Content Archetype Clustering**

1. **What one row means for my lane**: one row is one pseudonymized content item's aggregated behavior profile for a single calendar month — i.e. `dim_content.content_hash_id` rolled up from the daily grain of `fact_content_daily_performance` (which is itself one row per report date × client × content item) to content-month grain. The archetype clustering runs on these content-month profiles, not on raw daily rows.
2. **Table(s) I'll use**: `dim_content` (static content attributes: word count, publish date, content type) joined to `fact_content_daily_performance` (daily GSC/GA4 performance, partitioned by `month=YYYY-MM`) on `content_hash_id` + `client_hash_id`. I'll also touch `dim_clients` to filter to clients with usable access (`access_profile = 'gsc_and_ga4'`), since a content item from a GSC-only client has no engagement signal to cluster on.
3. **Time window**: a single mid-panel month, `month=2026-03`. The final month (`month=2026-06`, the `_sample` file) is a sealed test month — I never use it to develop or eyeball archetype logic, only (later) to check the clusters still hold up out-of-sample.
4. **What I'd predict/rank (label or proxy)**: there's no ground-truth archetype label in the warehouse, so this is unsupervised — the proxy target is a cluster assignment (cluster ID, e.g. via k-means on the content-month feature vector) learned from each content item's behavioral signature. Downstream I'll treat cluster membership as the thing I'm producing, not predicting against a known answer.
5. **One thing I deliberately exclude**: `fact_content_query_90d` (query-level keyword data). It's a different grain (client × content × query hash, fixed 90-day window) that doesn't align cleanly with a single calendar month, and pulling it in now would mean either mixing time windows or leaking the 90-day window's forward-looking coverage into a month-level clustering pass. I'm holding it in reserve for a later query-mix feature, not this contract.

In [3]:
from huggingface_hub import list_repo_files

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [5]:
# --- Setup: connect DuckDB to the gated HF warehouse ---
# pip install duckdb --quiet  # uncomment if duckdb isn't already installed in this runtime
import duckdb
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')  # Colab Secret — never paste the token directly into a cell
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MID_MONTH = "2026-03"       # mid-panel month — iterate here
FINAL_MONTH = "2026-06"     # sealed test month — never used for label logic

# --- Prove the grain: this DOUBLES as your first verification query ---
print("dim_content schema:")
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/dim_content.parquet') LIMIT 0"))

print("\nfact_content_daily_performance schema:")
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/*.parquet') LIMIT 0"))

print("\ndim_clients schema (for reference — already partly confirmed):")
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/dim_clients.parquet') LIMIT 0"))

# Run this cell first. Compare the printed column names against every
# `# VERIFY:` comment below and fix any that don't match before continuing.

dim_content schema:
┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Field | Why |
|---|---|---|
| **Feature** | `gsc_clicks`, `gsc_impressions` *(VERIFY names)* | Raw GSC counts for the month — base signal for CTR/position features |
| **Feature** | `gsc_avg_position` *(VERIFY)* | Ranking behavior, distinguishes archetypes (e.g. head-term vs long-tail pages) |
| **Feature** | `ga4_sessions`, `ga4_engagement_rate` *(VERIFY)* | On-site behavior signal, separate from search visibility |
| **Feature** | `dim_content.word_count` *(VERIFY)* | Static content attribute, knowable at any decision moment |
| **Feature** | `dim_content.publish_date` *(VERIFY)* | Used to derive content age — static, knowable |
| **Label (proxy)** | *(none stored)* — k-means cluster ID computed downstream | Unsupervised: the cluster assignment IS the output, not a stored column |
| **Context** | `client_hash_id`, `content_hash_id`, `month` | Keys/partitioning — not fed into the clustering model itself |
| **Context** | `dim_clients.access_profile` | Used to filter to clients with usable GSC+GA4 coverage, not as a feature |
| **Excluded** | `fact_content_query_90d.*` | Different grain (90-day window, query-level) — see Section 1, point 5 |
| **Excluded** | Any row where `month = '2026-06'` (`_sample`) | Sealed test month — reserved for later out-of-sample check, never for developing cluster logic |

In [7]:
# Quick eyeball of real values before trusting the table above
con.sql(f"SELECT * FROM read_parquet('{REL}/dim_content.parquet') LIMIT 5").show()
con.sql(f"SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet') LIMIT 5").show()

┌─────────────────────────┬──────────────────────────┬──────────────────────────┬──────────────────────┬────────────────────┬─────────────────────┬────────────────┬──────────────────────┬──────────────────────┬─────────────────┬───────────────┬─────────────┬───────────────────┬────────┬───────────────┬───────────┬────────────────┬──────────────────────┬─────────────────────────┬────────────────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────┬──────────────┬────────────┐
│     client_hash_id      │     content_hash_id      │     keyword_hash_id      │     url_hash_id      │ keyword_char_count │ keyword_token_count │ url_char_count │ content_created_date │ content_updated_date │  content_type   │ search_volume │ competition │ competition_level │  cpc   │  main_intent  │ backlinks │ category_count │ keyword_created_date │      provider_used      │       model_used       │ char_count │ word_count │ last_optimized_date │ optimization_eligible_date │ is_p

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Fact 1 — the grain**: one row of `fact_content_daily_performance` really is one (report date, client, content item) — no duplicates.

**Fact 2 — slice size + date span**: for `month=2026-03`, filtered to clients with full `gsc_and_ga4` access, how many rows, how many distinct content items, and what date range.

**Fact 3 — availability**: filtering with `IS TRUE` on the access flag, how many rows survive vs. the unfiltered count.

In [8]:
# --- Fact 1: grain check — zero rows back means the grain claim holds ---
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet')
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()
print(f"Duplicate (date, client, content) combinations: {len(grain_check)}  <- should be 0")

# --- Fact 2: slice row count, distinct content items, date span ---
slice_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT f.content_hash_id) AS distinct_content_items,
        MIN(f.report_date) AS earliest_date,
        MAX(f.report_date) AS latest_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet') f
    JOIN read_parquet('{REL}/dim_clients.parquet') c USING (client_hash_id)
    WHERE c.access_profile = 'gsc_and_ga4'
""").df()
print(slice_summary)

# --- Fact 3: availability, filtered with IS TRUE ---
before = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet')").df()['n'][0]
after = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet') f
    JOIN read_parquet('{REL}/dim_clients.parquet') c USING (client_hash_id)
    WHERE c.has_ga4_access IS TRUE
""").df()['n'][0]
print(f"Rows before availability filter: {before}")
print(f"Rows after has_ga4_access IS TRUE: {after}  ({after/before:.1%} survive)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (date, client, content) combinations: 0  <- should be 0
   row_count  distinct_content_items earliest_date latest_date
0    7700646                  260616    2026-03-01  2026-03-31
Rows before availability filter: 9841378
Rows after has_ga4_access IS TRUE: 7700646  (78.2% survive)


## 3b. Five features (max)

Built from the same mid-panel month, at content-item grain. Each one states why it's knowable at the decision moment (i.e. doesn't reach into the future).

In [12]:
print("=== dim_content ===")
for row in con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/dim_content.parquet')").fetchall():
    print(row)

print("\n=== fact_content_daily_performance ===")
for row in con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet')").fetchall():
    print(row)

=== dim_content ===
('client_hash_id', 'VARCHAR', 'YES', None, None, None)
('content_hash_id', 'VARCHAR', 'YES', None, None, None)
('keyword_hash_id', 'VARCHAR', 'YES', None, None, None)
('url_hash_id', 'VARCHAR', 'YES', None, None, None)
('keyword_char_count', 'BIGINT', 'YES', None, None, None)
('keyword_token_count', 'BIGINT', 'YES', None, None, None)
('url_char_count', 'BIGINT', 'YES', None, None, None)
('content_created_date', 'DATE', 'YES', None, None, None)
('content_updated_date', 'DATE', 'YES', None, None, None)
('content_type', 'VARCHAR', 'YES', None, None, None)
('search_volume', 'BIGINT', 'YES', None, None, None)
('competition', 'DOUBLE', 'YES', None, None, None)
('competition_level', 'VARCHAR', 'YES', None, None, None)
('cpc', 'DOUBLE', 'YES', None, None, None)
('main_intent', 'VARCHAR', 'YES', None, None, None)
('backlinks', 'BIGINT', 'YES', None, None, None)
('category_count', 'BIGINT', 'YES', None, None, None)
('keyword_created_date', 'DATE', 'YES', None, None, None)
('p

In [13]:
# --- Fact 1: grain check — zero rows back means the grain claim holds ---
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet')
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()
print(f"Duplicate (date, client, content) combinations: {len(grain_check)}  <- should be 0")

# --- Fact 2: slice row count, distinct content items, date span ---
slice_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT f.content_hash_id) AS distinct_content_items,
        MIN(f.report_date) AS earliest_date,
        MAX(f.report_date) AS latest_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet') f
    JOIN read_parquet('{REL}/dim_clients.parquet') c ON f.client_hash_id = c.client_hash_id
    WHERE c.access_profile = 'gsc_and_ga4'
""").df()
print(slice_summary)

# --- Fact 3: availability, filtered with IS TRUE (row-level flag, not client-level) ---
before = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet')").df()['n'][0]
after = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet')
    WHERE ga4_data_available IS TRUE
""").df()['n'][0]
print(f"Rows before availability filter: {before}")
print(f"Rows after ga4_data_available IS TRUE: {after}  ({after/before:.1%} survive)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (date, client, content) combinations: 0  <- should be 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count  distinct_content_items earliest_date latest_date
0    7700646                  260616    2026-03-01  2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows before availability filter: 9841378
Rows after ga4_data_available IS TRUE: 413966  (4.2% survive)


In [14]:
feature_frame = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS avg_ctr,
        AVG(f.gsc_avg_position) AS avg_position,
        SUM(f.ga4_sessions) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS sessions_per_impression,
        DATE_DIFF('day', MAX(d.content_created_date), DATE '{MID_MONTH}-01') AS content_age_days,
        MAX(d.word_count) AS word_count
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet') f
    JOIN read_parquet('{REL}/dim_content.parquet') d ON f.content_hash_id = d.content_hash_id
    JOIN read_parquet('{REL}/dim_clients.parquet') c ON f.client_hash_id = c.client_hash_id
    WHERE c.access_profile = 'gsc_and_ga4'
    GROUP BY 1, 2
""").df()
feature_frame.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,avg_ctr,avg_position,sessions_per_impression,content_age_days,word_count
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,0.001754,4.394234,0.000000,366,<NA>
1,content_a7da352b73b02668,client_73cda7b4e4f265ea,0.002629,7.244844,0.000405,366,2330
2,content_d056587ff7faca0c,client_73cda7b4e4f265ea,0.005776,4.459107,0.001083,366,2475
3,content_bfd1e41c2af250c8,client_73cda7b4e4f265ea,0.000000,14.753175,0.000000,366,<NA>
4,content_2662845f598544ef,client_73cda7b4e4f265ea,0.006667,6.341880,0.006667,366,<NA>
5,content_f39be42b42a4e8f6,client_73cda7b4e4f265ea,0.000000,14.432540,0.166667,366,<NA>
6,content_1855a661b4d36130,client_73cda7b4e4f265ea,0.002331,4.209227,0.004662,366,<NA>
7,content_22c063002b7c1caf,client_73cda7b4e4f265ea,0.003185,9.155335,0.000000,366,<NA>
8,content_0ea64f25303c9a77,client_73cda7b4e4f265ea,0.010101,23.025154,0.010101,366,<NA>
9,content_d720dde3701523c0,client_73cda7b4e4f265ea,0.007576,24.308216,0.000000,366,<NA>


## 3c. The trap — deliberate leakage

Add one column derived from the sealed test month (`month=2026-06`), watch a quick downstream score jump toward perfect, then delete it and keep the honest number. This is the notebook-02 leakage lesson, performed here on real warehouse data.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# A proxy "high performer" label for this quick check only (median split on avg_ctr)
ff = feature_frame.dropna(subset=['avg_ctr', 'avg_position', 'sessions_per_impression', 'content_age_days', 'word_count']).copy()
ff['high_performer'] = (ff['avg_ctr'] > ff['avg_ctr'].median()).astype(int)

honest_features = ['avg_position', 'sessions_per_impression', 'content_age_days', 'word_count']

# --- LEAK: pull in the sealed final month's clicks as a "feature" ---
leak = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_clicks) AS future_clicks
    FROM read_parquet('{REL}/fact_content_daily_performance/month={FINAL_MONTH}/*.parquet')
    GROUP BY 1
""").df()
ff_leaked = ff.merge(leak, on='content_hash_id', how='left').fillna({'future_clicks': 0})

def quick_auc(df, feature_cols):
    X = df[feature_cols]
    y = df['high_performer']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

leaked_auc = quick_auc(ff_leaked, honest_features + ['future_clicks'])
print(f"AUC WITH leaked future_clicks column: {leaked_auc:.3f}  <- jumps toward 1.0, this is the trap")

# --- Delete the leak, keep the honest number ---
honest_auc = quick_auc(ff, honest_features)
print(f"AUC WITHOUT the leak (honest):        {honest_auc:.3f}  <- this is the number that survives")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

AUC WITH leaked future_clicks column: 0.826  <- jumps toward 1.0, this is the trap
AUC WITHOUT the leak (honest):        0.657  <- this is the number that survives


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**One named limitation of this slice**: the warehouse is an unbalanced panel — `dim_clients.gsc_data_start` / `ga4_data_start` differ per client, so a content item's "month" of data can mean different things depending on which client it belongs to (a client onboarded mid-month has a partial month, not a missing one). Filtering to `access_profile = 'gsc_and_ga4'` addresses the *access* gap, but it does not fix the *history-depth* gap — two clients can both have full access today while one has only 3 months of history and the other has 18. Any archetype cluster built on `month=2026-03` alone can't distinguish "consistently this type of content" from "this client just started being tracked this type of way," and that distinction matters before treating a cluster as a stable behavioral pattern rather than a one-month snapshot.

In [17]:
# Evidence for the limitation above: history-depth spread across clients in this slice
history_spread = con.sql(f"""
    SELECT
        MIN(gsc_data_start) AS earliest_client_start,
        MAX(gsc_data_start) AS latest_client_start,
        DATE_DIFF('day', MIN(gsc_data_start), MAX(gsc_data_start)) AS spread_days
    FROM read_parquet('{REL}/dim_clients.parquet')
    WHERE access_profile = 'gsc_and_ga4'
""").df()  # VERIFY: gsc_data_start column name
print(history_spread)

  earliest_client_start latest_client_start  spread_days
0            2025-01-27          2026-06-02          491


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.